In [1]:
pip install cryptography pillow


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: cryptography in c:\users\aarth\appdata\local\programs\python\python312\lib\site-packages (46.0.3)




[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import json
import base64
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

In [2]:
# ============================================================
MASTER_KEY = b"\x01" * 32  # SAME key from encryption file

In [3]:
# ============================================================
# Helper: Base64 decode
# ============================================================
def b64(data):
    return base64.b64decode(data)

In [4]:
# ============================================================
# 1. Unwrap CEK using MASTER KEY (AES-GCM)
# ============================================================
def unwrap_cek(wrapped_cek, cek_nonce, cek_tag, master_key):
    aesgcm = AESGCM(master_key)

    # GCM expects ciphertext + tag combined
    encrypted_cek = wrapped_cek + cek_tag

    cek = aesgcm.decrypt(cek_nonce, encrypted_cek, None)
    return cek

In [5]:
# ============================================================
# 2. Decrypt image using CEK (AES-GCM)
# ============================================================
def decrypt_image(ciphertext, nonce, tag, cek):
    aesgcm = AESGCM(cek)
    full_ct = ciphertext + tag
    plaintext = aesgcm.decrypt(nonce, full_ct, None)
    return plaintext

In [9]:
# ============================================================
# MAIN PIPELINE
# ============================================================
def decrypt_from_json(json_file, output_image):
    # Step 1 — Load encrypted package
    with open(json_file, "r") as f:
        data = json.load(f)

    # Step 2 — Decode base64 values
    ciphertext = b64(data["ciphertext"])
    nonce = b64(data["nonce"])
    tag = b64(data["tag"])

    wrapped_cek = b64(data["wrapped_cek"])
    cek_nonce = b64(data["cek_nonce"])
    cek_tag = b64(data["cek_tag"])

    # Step 3 — Unwrap CEK
    cek = unwrap_cek(wrapped_cek, cek_nonce, cek_tag, MASTER_KEY)

    # Step 4 — Decrypt image
    plaintext_image = decrypt_image(ciphertext, nonce, tag, cek)

    # Step 5 — Save restored image
    with open(output_image, "wb") as f:
        f.write(plaintext_image)

    print("Decryption complete →", output_image)


In [10]:
# ============================================================
# Execute Example
# ============================================================
if __name__ == "__main__":
    decrypt_from_json("encrypted_output.json", "recovered_output.jpg")

Decryption complete → recovered_output.jpg
